# Nivel 2 — LangChain + Gemini + Telegram + Tools de Stock/Pedidos

Este notebook reimplementa el mismo caso de uso del flujo visual de N8n en una solución programática con **LangChain**, **Gemini**, **Telegram Bot** y herramientas para **stock, pedidos y facturas**.

## Objetivo
- Recibir mensajes desde Telegram.
- Clasificar intención con Gemini.
- Crear pedidos o consultar estado.
- Leer y actualizar las hojas **Stock**, **Pedidos** y **Facturas**.
- Responder al usuario por Telegram.

## Secrets requeridos
- `GOOGLE_API_KEY`
- `TELEGRAM_BOT_TOKEN`

## Archivo de datos
Este notebook usa `Registros.xlsx`, con las hojas **Pedidos**, **Stock** y **Facturas**.

In [20]:
!pip install -q -U langchain langchain-core langchain-google-genai python-telegram-bot pandas==2.2.2 openpyxl pydantic==2.12.3
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## 1. Cargar credenciales

En Google Colab, guarda las llaves en **Secrets**. Si ejecutas localmente, también puedes definirlas como variables de entorno.

In [21]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    print("✅ GOOGLE_API_KEY cargada desde Colab Secrets")
    print("✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets")
except Exception:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
    if os.getenv("GOOGLE_API_KEY"):
        print("✅ GOOGLE_API_KEY cargada desde variables de entorno")
    else:
        print("⚠️ Falta GOOGLE_API_KEY")
    if TELEGRAM_BOT_TOKEN:
        print("✅ TELEGRAM_BOT_TOKEN cargado desde variables de entorno")
    else:
        print("⚠️ Falta TELEGRAM_BOT_TOKEN")

✅ GOOGLE_API_KEY cargada desde Colab Secrets
✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets


## 2. Configuración general

In [35]:
from pathlib import Path
import pandas as pd
from datetime import datetime
from uuid import uuid4

EXCEL_PATH = Path("Registros.xlsx")
TZ_LABEL = "America/Bogota"

assert EXCEL_PATH.exists(), f"No se encontró {EXCEL_PATH.resolve()}"
print(f"✅ Archivo encontrado: {EXCEL_PATH.resolve()}")

✅ Archivo encontrado: /content/Registros.xlsx


## 3. Cargar hojas y revisar estructura

El archivo entregado contiene una tabla `Pedidos`, 30 productos en `Stock` ; además una tabla `Factura` , `Stock` usa las columnas `producto_id`, `descripcion_producto`, `stock` y `precio_unitario`, mientras `Facturas` usa `factura_id`, `order_id`, `cliente`, `monto_total`, `fecha_factura` y `estado_factura`.

In [36]:
stock_df = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
pedidos_df = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
facturas_df = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

print("Stock:", stock_df.shape)
print("Pedidos:", pedidos_df.shape)
print("Facturas:", facturas_df.shape)

stock_df.head()

Stock: (30, 4)
Pedidos: (0, 11)
Facturas: (0, 6)


,producto_id,descripcion_producto,stock,precio_unitario
0,PROD-001,Caja de guantes industriales talla M,120,18000
1,PROD-002,Cinta de embalaje 48 mm x 100 m,85,9500
2,PROD-003,Rollo de etiqueta térmica 100x100,38,32000
3,PROD-004,Lector de código de barras inalámbrico,12,145000
4,PROD-005,Impresora térmica de etiquetas,6,420000


## 4. Funciones auxiliares para Excel

Estas funciones simulan las tools del agente sobre el archivo `Registros.xlsx`.

In [37]:
EXPECTED_PEDIDOS = [
    "order_id", "cliente", "chat_id", "producto_id", "descripcion_producto",
    "cantidad", "estado", "stock", "fecha_pedido", "fecha_despacho", "total"
]
EXPECTED_STOCK = ["producto_id", "descripcion_producto", "stock", "precio_unitario"]
EXPECTED_FACTURAS = ["factura_id", "order_id", "cliente", "monto_total", "fecha_factura", "estado_factura"]


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def load_sheets():
    stock = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
    pedidos = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
    facturas = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

    for col in EXPECTED_PEDIDOS:
        if col not in pedidos.columns:
            pedidos[col] = ""
    for col in EXPECTED_STOCK:
        if col not in stock.columns:
            stock[col] = ""
    for col in EXPECTED_FACTURAS:
        if col not in facturas.columns:
            facturas[col] = ""

    return stock[EXPECTED_STOCK].copy(), pedidos[EXPECTED_PEDIDOS].copy(), facturas[EXPECTED_FACTURAS].copy()


def save_sheets(stock, pedidos, facturas):
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as writer:
        pedidos.to_excel(writer, sheet_name="Pedidos", index=False)
        stock.to_excel(writer, sheet_name="Stock", index=False)
        facturas.to_excel(writer, sheet_name="Facturas", index=False)

## 5. Tools del caso logístico

El flujo N8n del proyecto hace exactamente estas operaciones: clasifica intención, registra pedido, consulta `Stock`, valida disponibilidad, actualiza `Pedidos`, genera `Facturas` y notifica por Telegram.

In [38]:
from typing import Optional, Literal
from pydantic import BaseModel, Field

class IntentOutput(BaseModel):
    intencion: Literal["crear_pedido", "consultar_estado", "saludo", "otro"]
    order_id: str = ""
    producto_id: str = ""
    cantidad: int = 0


def get_stock(producto_id: str) -> dict:
    stock, _, _ = load_sheets()
    row = stock[stock["producto_id"].astype(str).str.upper() == producto_id.upper()]
    if row.empty:
        return {"found": False, "producto_id": producto_id, "mensaje": "Producto no encontrado"}
    r = row.iloc[0]
    return {
        "found": True,
        "producto_id": str(r["producto_id"]),
        "descripcion_producto": str(r["descripcion_producto"]),
        "stock": int(r["stock"]),
        "precio_unitario": float(r["precio_unitario"]),
    }


def create_order(cliente: str, chat_id: str, producto_id: str, cantidad: int) -> dict:
    stock, pedidos, facturas = load_sheets()
    info = get_stock(producto_id)
    order_id = f"{cliente}{str(uuid4())[:8]}"
    fecha_pedido = now_str()

    if not info["found"]:
        new_row = {
            "order_id": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": "",
            "cantidad": int(cantidad),
            "estado": "PRODUCTO_NO_ENCONTRADO",
            "stock": 0,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": "",
            "total": 0,
        }
        pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
        save_sheets(stock, pedidos, facturas)
        return {"ok": False, "order_id": order_id, "estado": "PRODUCTO_NO_ENCONTRADO", "mensaje": "Producto no encontrado"}

    stock_actual = int(info["stock"])
    precio_unitario = float(info["precio_unitario"])
    total = int(cantidad * precio_unitario)

    if stock_actual >= cantidad:
        stock.loc[stock["producto_id"].astype(str).str.upper() == producto_id.upper(), "stock"] = stock_actual - cantidad
        fecha_despacho = now_str()
        new_row = {
            "order_id": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "estado": "DESPACHADO",
            "stock": stock_actual,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
            "total": total,
        }
        pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
        factura = {
            "factura_id": f"FAC-{order_id}",
            "order_id": order_id,
            "cliente": cliente,
            "monto_total": total,
            "fecha_factura": fecha_despacho,
            "estado_factura": "GENERADA",
        }
        facturas = pd.concat([facturas, pd.DataFrame([factura])], ignore_index=True)
        save_sheets(stock, pedidos, facturas)
        return {
            "ok": True,
            "order_id": order_id,
            "estado": "DESPACHADO",
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "stock_disponible": stock_actual,
            "precio_unitario": precio_unitario,
            "total": total,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
        }

    new_row = {
        "order_id": order_id,
        "cliente": cliente,
        "chat_id": str(chat_id),
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "estado": "SIN_STOCK",
        "stock": stock_actual,
        "fecha_pedido": fecha_pedido,
        "fecha_despacho": now_str(),
        "total": 0,
    }
    pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
    save_sheets(stock, pedidos, facturas)
    return {
        "ok": False,
        "order_id": order_id,
        "estado": "SIN_STOCK",
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "stock_disponible": stock_actual,
        "total": 0,
    }


def get_order_status(chat_id: str, order_id: Optional[str] = None) -> dict:
    _, pedidos, _ = load_sheets()
    pedidos["chat_id"] = pedidos["chat_id"].astype(str)
    rows = pedidos[pedidos["chat_id"] == str(chat_id)].copy()
    if order_id:
        rows = rows[rows["order_id"].astype(str) == str(order_id)]
    rows = rows[rows["estado"].astype(str) != "SIN_STOCK"]
    if rows.empty:
        return {"ok": False, "mensaje": "No encontré pedidos válidos para consultar el estado."}
    last = rows.iloc[-1].to_dict()
    return {"ok": True, **last}

## 6. Modelo Gemini + Prompt + salida estructurada

El flujo N8n usa Gemini con un parser estructurado que obliga a devolver `intencion`, `order_id`, `producto_id` y `cantidad`; esa misma idea se replica aquí con LangChain y Pydantic.

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

structured_llm = llm.with_structured_output(IntentOutput)

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Eres un asistente de logística que recibe mensajes por Telegram.
Debes identificar la intención del usuario y devolver únicamente un objeto estructurado.

Reglas:
- intencion solo puede ser: crear_pedido, consultar_estado, saludo, otro.
- Si el usuario quiere comprar o pedir un producto, usa crear_pedido.
- Si el usuario pregunta por el estado de un pedido, usa consultar_estado.
- Si solo saluda, usa saludo.
- Si no aplica ninguna, usa otro.
- producto_id debe verse como PROD-XXX si aparece.
- cantidad debe ser numérica.
- Si falta información, deja el campo vacío o en 0.
"""),
    ("human", "Mensaje del usuario: {mensaje}")
])

classifier_chain = prompt | structured_llm

## 7. Lógica del asistente

In [40]:
def build_response(parsed: IntentOutput, cliente: str, chat_id: str) -> str:
    if parsed.intencion == "saludo":
        return (
            "Hola, soy tu asistente de logística. "
            "Puedo ayudarte a crear pedidos y consultar el estado de tus pedidos."
        )

    if parsed.intencion == "crear_pedido":
        if not parsed.producto_id or int(parsed.cantidad or 0) <= 0:
            return (
                "Para crear el pedido necesito un producto y una cantidad. "
                "Ejemplo: quiero pedir PROD-003 cantidad 2"
            )
        result = create_order(cliente=cliente, chat_id=chat_id, producto_id=parsed.producto_id, cantidad=int(parsed.cantidad))
        if result.get("estado") == "DESPACHADO":
            return (
                f"""Tu pedido {result['order_id']} fue procesado correctamente.
Producto: {result['descripcion_producto']}
Cantidad: {result['cantidad']}
Total: {result['total']}
Estado: DESPACHADO"""
            )
        if result.get("estado") == "SIN_STOCK":
            return (
                f"""Tu pedido {result['order_id']} no pudo procesarse.
Producto: {result['descripcion_producto']}
Cantidad solicitada: {result['cantidad']}
Stock disponible: {result['stock_disponible']}
Estado: SIN_STOCK"""
            )
        return "No se encontró el producto solicitado."

    if parsed.intencion == "consultar_estado":
        result = get_order_status(chat_id=chat_id, order_id=parsed.order_id or None)
        if not result.get("ok"):
            return result["mensaje"]
        return (
            f"""Pedido {result['order_id']}
Producto: {result.get('descripcion_producto') or result.get('producto_id')}
Cantidad: {result['cantidad']}
Estado: {result['estado']}
Total: {result.get('total', 0)}
Fecha pedido: {result.get('fecha_pedido', 'No registrada')}
Fecha despacho: {result.get('fecha_despacho', 'Pendiente')}"""
        )

    return (
        """No entendí tu solicitud. Puedes escribir algo como:
- quiero pedir PROD-002 cantidad 2
- consultar estado de mi pedido"""
    )


def process_message(mensaje: str, cliente: str = "Daniel", chat_id: str = "6172774306"):
    parsed = classifier_chain.invoke({"mensaje": mensaje})
    answer = build_response(parsed, cliente=cliente, chat_id=str(chat_id))
    return parsed, answer


## 8. Pruebas locales del agente

In [41]:
parsed, answer = process_message("quiero pedir PROD-003 cantidad 2")
print(parsed)
print("---")
print(answer)

intencion='crear_pedido' order_id='' producto_id='PROD-003' cantidad=2
---
Tu pedido Daniel2171879c fue procesado correctamente.
Producto: Rollo de etiqueta térmica 100x100
Cantidad: 2
Total: 64000
Estado: DESPACHADO


In [42]:
parsed, answer = process_message("consultar estado de mi pedido")
print(parsed)
print("---")
print(answer)

intencion='consultar_estado' order_id='' producto_id='' cantidad=0
---
Pedido Daniel2171879c
Producto: Rollo de etiqueta térmica 100x100
Cantidad: 2
Estado: DESPACHADO
Total: 64000
Fecha pedido: 2026-06-03 22:15:35
Fecha despacho: 2026-06-03 22:15:35


## 9. Integración con Telegram Bot

El tutorial del curso ya muestra que el token del bot se crea con BotFather y luego se valida con `python-telegram-bot`; aquí esa misma credencial se usa para conectar Telegram al agente programático.

In [43]:
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes, MessageHandler, filters

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Hola, soy tu asistente de logística. Puedes escribir: quiero pedir PROD-003 cantidad 2"
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not update.message or not update.message.text:
        return

    mensaje = update.message.text
    cliente = update.effective_user.first_name or "Cliente"
    chat_id = str(update.effective_chat.id)

    try:
        parsed, answer = process_message(mensaje=mensaje, cliente=cliente, chat_id=chat_id)
        print("\n===== TRAZA DEL AGENTE =====")
        print("Mensaje:", mensaje)
        print("Estructura detectada:", parsed.model_dump())
        print("Respuesta final:", answer)
        await update.message.reply_text(answer)
    except Exception as e:
        await update.message.reply_text(f"Ocurrió un error procesando tu solicitud: {e}")


def build_telegram_app():
    app = ApplicationBuilder().token(TELEGRAM_BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    return app

## 10. Ejecutar el bot

Descomenta la última línea para dejar el bot corriendo en Colab. Si van a grabar video, esta es la parte ideal para mostrar la ejecución en tiempo real y la traza del agente.

In [44]:
app = build_telegram_app()
await app.initialize()
await app.start()
await app.updater.start_polling()

print("✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.")

✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.


In [45]:
await app.updater.stop()
await app.stop()
await app.shutdown()

print("🛑 Bot detenido.")

🛑 Bot detenido.


## 11. Ejemplo de evidencias para diapositivas 07–09

### Arquitectura
Telegram -> Handler Python -> ChatPromptTemplate -> Gemini -> Tools (Stock/Pedidos/Facturas) -> Respuesta al usuario.

### Tools implementadas
- `get_stock(producto_id)`
- `create_order(cliente, chat_id, producto_id, cantidad)`
- `get_order_status(chat_id, order_id)`

### Justificación técnica
- Gemini interpreta intención y extrae entidades.
- Las rules de negocio se ejecutan con tools determinísticas.
